# 04. YOLO11 (one-stage, anchor-free)

Ultralytics YOLO11s з вагами COCO. Один ноутбук на всі запуски, спільний код — у `hw2_ultra.py`. Оцінка — тим самим `hw2_eval.py` на повному val.

| Запуск | Що змінено | Тип |
|---|---|---|
| `baseline` | YOLO11s, 960 px, стандартні налаштування Ultralytics | — |
| `exp1_imgsz1280` | вхідна роздільність 960 → 1280 | гіперпараметр |
| `exp2_nwd` | втрата рамок: CIoU → 0,5·CIoU + 0,5·NWD, роздільність — краща з `baseline` / `exp1_imgsz1280` | функція втрат |

Під час навчання Ultralytics рахує свій mAP на кожному 5-му кадрі val (`val_sub.txt`), без ignored regions — лише для кривої навчання і вибору `best.pt`.

In [ ]:
EXPERIMENT = "baseline"
SMOKE_TEST = False

In [ ]:
EXPERIMENTS = {
    "baseline": {"model": "yolo11s.pt", "imgsz": 960, "batch": 8, "nwd": 0.0},
    "exp1_imgsz1280": {"model": "yolo11s.pt", "imgsz": 1280, "batch": 4, "nwd": 0.0},
    "exp2_nwd": {"model": "yolo11s.pt", "imgsz": "best", "batch": "best", "nwd": 0.5},
}

EPOCHS = 20
NWD_C = 3.0
SEED = 42
WORKERS = 16

## 1. Шляхи і дані

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    for name in ("hw2_eval.py", "hw2_ultra.py"):
        sys.path.append(str(next(Path("/kaggle/input").glob(f"**/{name}")).parent))
import hw2_eval as E
import hw2_ultra as U

DATA, PREPARED, RUNS = U.paths()
SETUP = dict(EXPERIMENTS[EXPERIMENT])
if SETUP["imgsz"] == "best":
    best = EXPERIMENTS[U.best_run(RUNS / "yolo", ["baseline", "exp1_imgsz1280"])]
    SETUP.update(imgsz=best["imgsz"], batch=best["batch"])
OUT = RUNS / "yolo" / (f"smoke_{EXPERIMENT}" if SMOKE_TEST else EXPERIMENT)
print(U.gpu_name(), SETUP, OUT, sep="\n")

In [ ]:
val = U.val_frames(DATA)
test = U.test_frames(DATA)
val_sub = val[val["pos"] % U.VAL_STEP == 0]
if SMOKE_TEST:
    EPOCHS = 1
    val, test, val_sub = val.head(4), test.head(4), val_sub.head(8)
data_yaml = U.write_data_yaml(PREPARED, list(val_sub["path"]), OUT)
pd.Series({"val (кожна епоха)": len(val_sub), "val (повний)": len(val), "test": len(test)})

## 2. Втрата рамок з NWD (експ. 2)

Ultralytics рахує втрату рамок як `1 − CIoU` у `BboxLoss`, викликаючи `bbox_iou` з модуля `ultralytics.utils.loss`. Підміняємо цю функцію лише в модулі втрат (assigner і метрики не змінюються) на суміш `(1 − w)·CIoU + w·NWD`, де NWD = exp(−W₂ / C), W₂ — відстань Вассерштейна між рамками як 2D-гауссіанами (arXiv:2110.13389). Рамки у втраті — в одиницях сітки (піксель / stride), тому C = 3: медіанний об'єкт ~23 px на кроці 8.

In [ ]:
import ultralytics.utils.loss as yolo_loss
from ultralytics.utils.metrics import bbox_iou


def nwd(box1, box2, c=NWD_C):
    center = ((box1[:, :2] + box1[:, 2:]) - (box2[:, :2] + box2[:, 2:])) / 2
    size = (box1[:, 2:] - box1[:, :2]) - (box2[:, 2:] - box2[:, :2])
    w2 = (center ** 2).sum(1, keepdim=True) + (size ** 2).sum(1, keepdim=True) / 4
    return torch.exp(-w2.clamp(min=1e-7).sqrt() / c)


def iou_with_nwd(box1, box2, xywh=False, **kwargs):
    iou = bbox_iou(box1, box2, xywh=xywh, **kwargs)
    return (1 - SETUP["nwd"]) * iou + SETUP["nwd"] * nwd(box1, box2)


yolo_loss.bbox_iou = iou_with_nwd if SETUP["nwd"] > 0 else bbox_iou

## 3. Навчання

Стандартний рецепт Ultralytics (оптимізатор `auto`, mosaic, hsv, flip, `close_mosaic=10`), `seed=42`, `deterministic=True`. Якщо запуск перервався, навчання продовжується з `last.pt`; якщо вже завершене — пропускається.

In [ ]:
elapsed = U.timer()
if U.resumable(OUT):
    YOLO(OUT / "weights" / "last.pt").train(resume=True)
elif not (OUT / "weights" / "last.pt").exists():
    YOLO(SETUP["model"]).train(data=str(data_yaml), imgsz=SETUP["imgsz"], batch=SETUP["batch"], epochs=EPOCHS, seed=SEED,
                               deterministic=True, workers=WORKERS, cache="disk", project=str(OUT.parent), name=OUT.name,
                               exist_ok=True, fraction=0.002 if SMOKE_TEST else 1.0, plots=True, verbose=False)
train_minutes = elapsed()
history = U.history(OUT)
history[["epoch", "mAP", "AP50"]].round(4).tail(5)

## 4. Повний val, швидкість, сабміт

Ваги `best.pt` (найкраща епоха за val Ultralytics). Поріг впевненості 0,001, NMS IoU 0,7, до 300 рамок.

In [ ]:
best_model = YOLO(OUT / "weights" / "best.pt")
config = {"experiment": EXPERIMENT, **SETUP, "nwd_c": NWD_C, "gpu": U.gpu_name(), "epochs": EPOCHS, "seed": SEED,
          "train_min": train_minutes}
summary, per_class, clips, speed = U.finish(best_model, OUT, config, SETUP["imgsz"], val, test, DATA)
print(f"Навчання: {train_minutes:.0f} хв, швидкість: {speed:.1f} мс/кадр")
display(summary.round(4), per_class.round(4), clips.round(4))

## Джерела

**Бібліотеки**
- Ultralytics 8.3.40 (AGPL-3.0) — YOLO11s, ваги COCO, навчання і передбачення: https://docs.ultralytics.com/
- faster-coco-eval через спільний `hw2_eval.py`.

**Код, взятий або адаптований**
- Формула втрати рамок `1 − IoU` з вагами і виклик `bbox_iou(..., CIoU=True)` — `ultralytics/utils/loss.py::BboxLoss.forward`; ми підміняємо `bbox_iou` лише в цьому модулі.
- NWD — формула з Wang et al., *A Normalized Gaussian Wasserstein Distance for Tiny Object Detection*, arXiv:2110.13389 (рівняння для W₂² між гауссіанами з центром рамки і коваріацією diag(w²/4, h²/4)).

**Статті**
- YOLOX (anchor-free YOLO), arXiv:2107.08430; TOOD (task-aligned assigner у YOLOv8/11), arXiv:2108.07755; GFL (distribution focal loss), arXiv:2006.04388; огляд Ultralytics YOLO, arXiv:2510.09653.
- CIoU — Zheng et al., arXiv:1911.08287.

**Наші рішення:** YOLO11s як компроміс швидкість/якість під 8 ГБ GPU, 20 епох, перевірка 1280 px через частку дрібних об'єктів з EDA, вага NWD 0,5 і C = 3 (у пікселях сітки), вибір роздільності для експ. 2 за mAP.